# bsc_02 — MVP Stage 1 (trục S/D/I/H chuẩn hóa)

Gate 1 đã **PROCEED** (`M0_gate1_results_and_decision.md`). Notebook chạy Stage 1 theo
plan §6 bước 3→13, dùng ma trận chuẩn hóa của `p3_m8d_standardization_decision_vi.md`.

**Trục cấu hình** (`bsc/experiment.py`):

| | | |
|---|---|---|
| **S** nguồn bề mặt | S0 GT · S1 GT+jitter · S2 dự đoán | |
| **D** miền khớp | D0 oracle · D1 atlas theo fold | |
| **I** kênh vào | I0 mri · I1 +grad · I2 +sdf · I3 +prob | |
| **H** đầu ra | H0 occupancy · H1 +presence | |

Ma trận §7: `P0=S0D0I0H0` · `P1=S0D0I1H0` · `P2=S0D1I2H1` · `P3=S0D1I3H1`
**P3 chính là M8-D** — một run, hai tên. Không train riêng.

**Thiết kế đánh giá:**
- Train/val từ **404 ca TRAIN**, fold theo `splits_zib_v1_fixed.json`. Train = fold 1–4,
  val = fold 0. **103 ca test không đụng tới** (chỉ mở lại ở Gate 4).
- Baseline = prediction **out-of-fold** 150ep (model không train trên ca đó).
- Atlas D1 xây **chỉ từ ca train của fold**, có `assert_no_leak` chặn vi phạm §2.2.
- Kênh `prob` = **out-of-fold softmax** (quyết định 2026-07-19: trung bình 5 fold rò rỉ
  trên ca train/val).

**Ràng buộc báo cáo (M0 §6):** không báo cáo ASSD tổng — mục tiêu quy về ASSD tổng chỉ
0.006mm, nhỏ ngang sai khác giữa hai implementation metric. Báo cáo ở **mẫu số vùng mỏng**
+ presence F1 + thickness MAE.


### Cell config chuẩn

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/bsCart-net.git"
REPO_DIR = "/content/repo"

import os, sys
if not os.path.isdir(f"{REPO_DIR}/bsc"):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!pip install -q nibabel SimpleITK 2>/dev/null

BSC_ROOT = "/content/drive/MyDrive/bsc"
for sub in ["splits","baselines","geom","raydb","atlas","runs"]:
    os.makedirs(f"{BSC_ROOT}/{sub}", exist_ok=True)
RAW = "/content/drive/MyDrive/nnUNet_raw/Dataset001_KneeOA"

from bsc import model, core, headroom, metrics, io_utils, atlas, experiment as X
from bsc.core import RayConfig
import numpy as np, glob, json
from tqdm import tqdm

cfg = RayConfig()
print("BSC_ROOT =", BSC_ROOT, "| prob mac dinh:", X.DEFAULT_PROB_SOURCE)

## 0. Kiểm tra setup — chạy TRƯỚC mọi thứ

Có dòng `THIEU` thì **dừng**, đừng chạy tiếp.

In [ ]:
ok = True
def chk(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"{'OK   ' if cond else 'THIEU'} | {name}{('  -> ' + detail) if detail else ''}")

n_img = len(glob.glob(f"{RAW}/imagesTr/*_0000.nii.gz"))
n_lab = len(glob.glob(f"{RAW}/labelsTr/oaizib_*.nii.gz"))
chk("imagesTr", n_img > 0, f"{n_img} anh")
chk("labelsTr", n_lab > 0, f"{n_lab} nhan")

SP = None
_labs = sorted(glob.glob(f"{RAW}/labelsTr/oaizib_*.nii.gz"))
if _labs:
    g, sp0 = io_utils.load_nii(_labs[0])
    have = set(int(v) for v in np.unique(g))
    chk("nhan co xuong+sun", {1,2,3,4}.issubset(have), f"co {sorted(have)}")

    # SPACING THAT cua dataset - dung cai nay o MOI NOI, KHONG dung core.SPACING
    SP = tuple(float(x) for x in sp0)
    same = np.allclose(SP, core.SPACING, atol=1e-3)
    chk("spacing", True, f"{tuple(round(x,4) for x in SP)}"
        + ("" if same else f"  !! KHAC core.SPACING {core.SPACING}"))
    if not same:
        print("       -> Truc 0.70mm KHONG o vi tri dau nhu core.SPACING gia dinh.")
        print("          Notebook nay truyen SP tuong minh khap noi. Neu ban goi ham nao")
        print("          ma QUEN truyen spacing, no se ap anisotropy SAI TRUC - loi im lang.")

    # Toan bo pipeline dung MOT SP chung => phai chac 404 ca cung spacing
    _bad = []
    for _p in _labs[::40]:
        _, _s = io_utils.load_nii(_p)
        if not np.allclose(_s, SP, atol=1e-3):
            _bad.append((os.path.basename(_p), tuple(round(x, 4) for x in _s)))
    chk("spacing dong nhat giua cac ca", not _bad,
        f"kiem {len(_labs[::40])} ca" if not _bad else f"LECH: {_bad[:3]}")
    if _bad:
        print("       -> Co ca khac spacing => KHONG dung mot SP chung duoc.")
        print("          Phai sua build_for/atlas de nhan spacing theo TUNG ca.")

SPLITS = f"{BSC_ROOT}/splits/splits_zib_v1_fixed.json"
chk("splits_zib_v1_fixed", os.path.exists(SPLITS))

CVP = glob.glob(f"{BSC_ROOT}/baselines/ds020/**/nnUNetTrainer_150epochs*/", recursive=True)
n_cv = len(glob.glob(f"{CVP[0]}/fold_*/validation/oaizib_*.nii.gz")) if CVP else 0
chk("CV 150ep predictions (baseline OOF)", n_cv > 0, f"{n_cv} file")

n_prob = len(glob.glob(f"{BSC_ROOT}/baselines/oof_prob/**/*.nii.gz", recursive=True))
print(f"{'OK   ' if n_prob else '(bo qua)'} | OOF softmax cho I3/P3  -> {n_prob} file"
      f"{'' if n_prob else '  (chua co: P0-P2 van chay duoc, chi P3 phai doi muc 1b)'}")

print("\n" + ("=> SAN SANG" if ok else "=> THIEU NGUYEN LIEU - xem dong THIEU"))

## 1. QC hình học trên XƯƠNG THẬT — plan §6 bước 3, 5, 6

M2/M3/M4 mới chỉ chạy phantom. Plan đòi chạy trên xương thật **trước** khi train.
M3 <99.5% hoặc M2 ASSD lớn ⇒ dừng, sửa hình học.

In [ ]:
CART = {"femoral_cart": 2, "med_tib_cart": 4}
BONE = {"femoral_cart": 1, "med_tib_cart": 3}
OPP  = {"femoral_cart": 3, "med_tib_cart": 1}     # xuong doi dien, cho atlas D1
QC_CKPT = f"{BSC_ROOT}/runs/mvp_geom_qc.jsonl"
N_QC = 10

cases_tr = sorted(os.path.basename(p)[:-len("_0000.nii.gz")]
                  for p in glob.glob(f"{RAW}/imagesTr/*_0000.nii.gz"))
cases_tr = [c for c in cases_tr if c.startswith("oaizib_")]
print(f"{len(cases_tr)} ca train")

done = set()
if os.path.exists(QC_CKPT):
    done = {(json.loads(l)["case"], json.loads(l)["cls"]) for l in open(QC_CKPT)}

with open(QC_CKPT, "a") as fh:
    for cid in tqdm(cases_tr[:N_QC], desc="geom QC"):
        gt, sp = io_utils.load_nii(f"{RAW}/labelsTr/{cid}.nii.gz")
        for c in CART:
            if (cid, c) in done: continue
            bone, cart = (gt == BONE[c]), (gt == CART[c])
            if not bone.any() or not cart.any(): continue
            sdf, verts, normals = core.bone_geometry(bone, sp, cfg)
            frac_ok, _ = core.check_normals(sdf, verts, normals, sp)
            occ = core.occupancy_target(cart, verts, normals, cfg, sp)
            rec = core.splat_rays(occ.astype(np.float32), verts, normals, cart.shape, cfg, sp)
            pred = rec > 0.5
            fr = atlas.fit_frame(verts)
            fh.write(json.dumps({
                "case": cid, "cls": c,
                "m3_normals_ok": float(frac_ok),
                "m4_single_interval": float(core.single_interval_ratio(occ)),
                "m2_dice": float(2*(pred & cart).sum()/(pred.sum()+cart.sum()+1e-8)),
                "m2_assd_mm": float(metrics.assd(cart, pred, sp)),
                "frame_degenerate": bool(fr.is_degenerate),
                "frame_extent_mm": [float(x) for x in fr.extent]}) + "\n")
            fh.flush()

rows = [json.loads(l) for l in open(QC_CKPT)]
print(f"\n{'lop':<14}{'M3 normals':>12}{'M4 single':>11}{'M2 Dice':>10}{'M2 ASSD':>10}{'khung suy bien':>16}")
for c in CART:
    r = [x for x in rows if x["cls"] == c]
    if not r: continue
    f = lambda k: np.mean([x[k] for x in r])
    print(f"{c:<14}{f('m3_normals_ok'):>11.1%}{f('m4_single_interval'):>11.1%}"
          f"{f('m2_dice'):>10.3f}{f('m2_assd_mm'):>9.3f}mm{f('frame_degenerate'):>15.0%}")

print("""
CONG (plan §3.5):
  M3 >99.5%  -> phap tuyen dung huong. Duoi nguong: KHONG train, sua hinh hoc truoc.
  M4 >95%    -> bieu dien theo bien kha thi. <90%: giu occupancy decoder.
  M2 dat cong tren ASSD (<0.1mm), KHONG phai Dice - xem core.roundtrip: sun 0.4mm cho
     Dice 0.848 du du doan HOAN HAO. Dice tren cau truc ~1 voxel nhay den muc tan nhan.
  khung suy bien >0% -> atlas D1 khong dung duoc cho nhung ca do (dung D0 thay the).""")

## 1b. (Tùy chọn) Sinh OOF softmax cho kênh I3 / P3

**Công tắc `RUN_OOF_PROB`** — mặc định `False` = bỏ qua hoàn toàn. Đổi thành `True` mới chạy.

**Chỉ cần nếu chạy P3/M8-D.** P0–P2 không dùng kênh này.

Cách hoạt động: với mỗi fold `k`, lấy các ca có `fold_of[case] == k` — đó là các ca fold `k`
dùng làm *validation*, nên model `k` **chưa từng train trên chúng** → softmax thu được là
**out-of-fold**, không rò rỉ nhãn vào kênh đầu vào.

**Vì sao chỉ giữ một kênh + float16:** lưu cả 8 lớp `float32` là ~750MB/ca → 300GB cho 404 ca.
Chỉ giữ lớp đích ở `float16` và nén (xác suất chủ yếu bằng 0 nên nén rất tốt) → ~5–20MB/ca.

Cell trích **cả hai lớp** (`femoral_cart` và `med_tib_cart`) trong **một lần inference** —
chạy lại chỉ để lấy lớp thứ hai là lãng phí 2–4 tiếng GPU.

⚠️ Cần **GPU**. Ước tính 404 ca: A100 ~40–70 phút, T4 ~2–4 tiếng. Có resume theo từng file.

In [ ]:
PROB_DIR = f"{BSC_ROOT}/baselines/oof_prob"
RUN_OOF_PROB = False        # <-- doi True khi muon chay P3/M8-D (2-4h GPU)

# Trich CA HAI lop trong MOT lan inference - chay lai chi de lay lop thu hai la
# lang phi 2-4h GPU. Duong dan: {PROB_DIR}/{ten_lop}/{case}.nii.gz
PROB_CLASSES = {"femoral_cart": 2, "med_tib_cart": 4}
for _c in PROB_CLASSES:
    os.makedirs(f"{PROB_DIR}/{_c}", exist_ok=True)

def prob_path(cls, cid):
    return f"{PROB_DIR}/{cls}/{cid}.nii.gz"

if RUN_OOF_PROB:
    import shutil
    os.environ["nnUNet_results"] = f"{BSC_ROOT}/baselines/ds020"
    os.environ["nnUNet_raw"] = os.path.dirname(RAW)
    os.environ["nnUNet_preprocessed"] = "/content/nnunet_prep_tmp"
    os.makedirs(os.environ["nnUNet_preprocessed"], exist_ok=True)
    import importlib.util
    if importlib.util.find_spec("nnunetv2") is None:
        !pip install -q nnunetv2

    # --- DANG KY TRAINER TU DINH NGHIA -------------------------------------------
    # nnUNetTrainer_150epochs / _250epochs KHONG co trong ban nnunetv2 cai tu pip
    # => "RuntimeError: Could not find requested nnunet trainer".
    # Chung chi la bien the doi SO EPOCH; khi INFERENCE trainer chi duoc dung de dung
    # KIEN TRUC MANG, ma kien truc do y het nnUNetTrainer goc => subclass rong la du.
    # Ghi file vao chinh thu muc trainer cua package de recursive_find tim thay.
    import nnunetv2, pathlib, importlib
    _trdir = pathlib.Path(nnunetv2.__file__).parent / "training" / "nnUNetTrainer"
    _tr_src = [
        "from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer",
        "",
        "",
        "class nnUNetTrainer_150epochs(nnUNetTrainer):",
        "    def __init__(self, *a, **kw):",
        "        super().__init__(*a, **kw)",
        "        self.num_epochs = 150",
        "",
        "",
        "class nnUNetTrainer_250epochs(nnUNetTrainer):",
        "    def __init__(self, *a, **kw):",
        "        super().__init__(*a, **kw)",
        "        self.num_epochs = 250",
        "",
    ]
    (_trdir / "bsc_custom_trainers.py").write_text(chr(10).join(_tr_src))
    importlib.invalidate_caches()
    from nnunetv2.utilities.find_objects import recursive_find_trainer_class_by_name as _find
    assert _find("nnUNetTrainer_150epochs") is not None, "dang ky trainer that bai"
    print("Da dang ky nnUNetTrainer_150epochs / _250epochs")

    fold_of_ = json.load(open(SPLITS))["fold_of"]
    for k in range(5):
        ids = [c for c in cases_tr if fold_of_.get(c) == k
               and not all(os.path.exists(prob_path(cl, c)) for cl in PROB_CLASSES)]
        if not ids:
            print(f"fold {k}: da du"); continue
        tmp_in, tmp_out = "/content/oof_in", "/content/oof_out"
        for d in (tmp_in, tmp_out):
            shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
        for c in ids:
            shutil.copy(f"{RAW}/imagesTr/{c}_0000.nii.gz", f"{tmp_in}/{c}_0000.nii.gz")
        print(f"fold {k}: {len(ids)} ca (OOF - model nay KHONG train tren chung)")
        !nnUNetv2_predict -i {tmp_in} -o {tmp_out} -d 020 -c 3d_fullres -tr nnUNetTrainer_150epochs -p nnUNetResEncUNetLPlans -f {k} --save_probabilities

        for c in ids:
            npz = f"{tmp_out}/{c}.npz"
            if not os.path.exists(npz):
                print(f"  thieu {c}.npz - bo qua"); continue
            arr = np.load(npz)["probabilities"]        # [n_class, ...]
            ref = f"{RAW}/labelsTr/{c}.nii.gz"
            lab, _ = io_utils.load_nii(ref)
            for cl, idx in PROB_CLASSES.items():
                io_utils.save_nii(arr[idx].astype(np.float16), prob_path(cl, c), reference=ref)
            # KIEM TRUC: prob doc lai phai CUNG SHAPE voi nhan. Thu tu truc cua npz do
            # nnUNet quyet dinh, khong hien nhien khop io_utils - phai kiem, khong doan.
            back, _ = io_utils.load_nii(prob_path(list(PROB_CLASSES)[0], c))
            assert back.shape == lab.shape, (
                f"{c}: prob shape {back.shape} != nhan {lab.shape}. Thu tu truc npz khac "
                f"gia dinh cua io_utils - phai transpose truoc khi luu.")
            os.remove(npz)

    n_ok = len(glob.glob(f"{PROB_DIR}/{list(PROB_CLASSES)[0]}/*.nii.gz"))
    size_gb = sum(os.path.getsize(f) for f in glob.glob(f"{PROB_DIR}/**/*.nii.gz",
                                                       recursive=True)) / 1e9
    print(f"Xong: {n_ok} ca x {len(PROB_CLASSES)} lop, {size_gb:.2f} GB -> {PROB_DIR}")
else:
    print("Bo qua (RUN_OOF_PROB=False). P0-P2 chay duoc; chi P3/M8-D moi can muc nay.")
    print("De chay: doi RUN_OOF_PROB=True, bat GPU, uoc tinh 2-4h tren T4.")

## 2. Split + atlas theo fold (D1) — plan §2.2

Atlas xây **chỉ từ ca train của fold**. `assert_no_leak` biến vi phạm §2.2 thành lỗi.

In [ ]:
CLS = "femoral_cart"          # §3.1: femoral la lop DAN de debug pipeline
N_TRAIN_CASES = 40
RAYS_PER_CASE = 20000

fold_of = json.load(open(SPLITS))["fold_of"]
zib = [c for c in cases_tr if c in fold_of]
train_ids = [c for c in zib if fold_of[c] != 0][:N_TRAIN_CASES]
val_ids   = [c for c in zib if fold_of[c] == 0][:10]
print(f"train {len(train_ids)}  val {len(val_ids)}")

CV_DIR = glob.glob(f"{BSC_ROOT}/baselines/ds020/**/nnUNetTrainer_150epochs*/", recursive=True)[0]
def baseline_path(cid):
    p = glob.glob(f"{CV_DIR}/fold_*/validation/{cid}.nii.gz")
    return p[0] if p else None

def make_loader(cls, pred_bone=False):
    def loader(cid):
        gt, sp = io_utils.load_nii(f"{RAW}/labelsTr/{cid}.nii.gz")
        mri, _ = io_utils.load_nii(f"{RAW}/imagesTr/{cid}_0000.nii.gz")
        if pred_bone:
            pr, _ = io_utils.load_nii(baseline_path(cid))
            bone = (pr == BONE[cls])
        else:
            bone = (gt == BONE[cls])
        return mri, bone, (gt == CART[cls])
    return loader

loader = make_loader(CLS)

# --- Atlas D1: CHI tu ca train cua fold ---
ATLAS_PATH = f"{BSC_ROOT}/atlas/atlas_{CLS}_fold0.npz"
if os.path.exists(ATLAS_PATH):
    z = np.load(ATLAS_PATH, allow_pickle=True)
    ATLAS = atlas.ArticularAtlas(z["prob"], int(z["n_bins"]), float(z["lo"]),
                                 float(z["hi"]), tuple(z["case_ids"]), 0, None)
    print(f"Nap atlas co san: {len(ATLAS.case_ids)} ca")
else:
    ATLAS = atlas.build_articular_atlas(train_ids, loader, SP, cfg,
                                        n_bins=24, min_count=3, fold=0, verbose=False)
    np.savez_compressed(ATLAS_PATH, prob=ATLAS.prob, n_bins=ATLAS.n_bins,
                        lo=ATLAS.lo, hi=ATLAS.hi, case_ids=np.array(ATLAS.case_ids))
    print(f"Xay atlas tu {len(ATLAS.case_ids)} ca train -> {ATLAS_PATH}")

atlas.assert_no_leak(ATLAS, val_ids)      # §2.2 - phai KHONG no
print("assert_no_leak OK: khong ca val nao duoc dung xay atlas")

cov = np.isfinite(ATLAS.prob).mean()
print(f"Atlas: {cov:.1%} o luoi co du lieu, xac suat TB {np.nanmean(ATLAS.prob):.3f}")

## 3. Hàm dựng dataset theo RunConfig

Một chỗ duy nhất dịch `RunConfig` → tham số `build_dataset`, để không cấu hình lệch giữa
các run.

In [ ]:
def domain_for(run, verts, occ):
    """D0 = oracle (tu sun GT) | D1 = atlas theo fold."""
    if run.domain == "D0":
        return core.oracle_domain(verts, occ, cfg)
    return atlas.atlas_domain(verts, ATLAS, thr=0.2)

def build_for(run, ids, seed, **kw):
    """Dung (X, occ, presence) dung theo cau hinh cua run."""
    extra = {}
    if "prob" in run.channels:
        extra_loader = lambda cid: io_utils.load_nii(prob_path(CLS, cid))[0]
    Xs, os_, ps = [], [], []
    for i, cid in enumerate(ids):
        mri, bone, cart = (make_loader(CLS, pred_bone=(run.surface == "S2")))(cid)
        ev = {"prob": extra_loader(cid)} if "prob" in run.channels else None
        jit = dict(jitter_s_mm=0.5, jitter_theta_deg=10.0) if run.surface == "S1" else {}
        _, verts, dirs0 = core.bone_geometry(bone, SP, cfg)
        occ0 = core.occupancy_target(cart, verts, dirs0, cfg, SP)
        dom = domain_for(run, verts, occ0)
        Xc, oc, pc, _, _ = model.build_case_rays(
            mri, bone, cart, SP, cfg, domain=dom,
            channels=run.channels, extra_vols=ev, seed=seed + i, **jit, **kw)
        if len(Xc) == 0: continue
        if RAYS_PER_CASE and len(Xc) > RAYS_PER_CASE:
            r = np.random.default_rng(seed + i).choice(len(Xc), RAYS_PER_CASE, replace=False)
            Xc, oc, pc = Xc[r], oc[r], pc[r]
        Xs.append(Xc); os_.append(oc); ps.append(pc)
    return np.concatenate(Xs), np.concatenate(os_), np.concatenate(ps)

def train_eval(run, epochs=30, lr=3e-4):
    Xa, oa, pa = build_for(run, train_ids, seed=0)
    Xb, ob, pb = build_for(run, val_ids, seed=100)
    net = model.RayEncoder1D(in_channels=len(run.channels))
    hist = model.fit(net, Xa, oa, pa, epochs=epochs, batch_size=4096, lr=lr, seed=run.seed)
    op, pp = model.predict_rays(net, Xb)
    dice = float(2*((op>0.5)&ob.astype(bool)).sum()/((op>0.5).sum()+ob.sum()+1e-8))
    f1 = metrics.presence_f1(pb.astype(bool), pp > 0.5)
    return {"net": net, "run": run, "val_occ_dice": dice, "presence": f1,
            "n_train_rays": int(len(Xa)), "hist": hist}

print("San sang.")

## 4. M5 — tiny-set overfitting trên dữ liệu THẬT (plan §3.5)

Trượt ⇒ có **bug** ở input/target/kiến trúc/loss. Đừng chỉnh hyperparameter.

In [ ]:
r_m5 = X.from_plan("P1", CLS, seed=1)
Xa, oa, pa = build_for(r_m5, train_ids[:4], seed=0)
i = np.random.default_rng(0).choice(len(Xa), min(4000, len(Xa)), replace=False)

net5 = model.RayEncoder1D(in_channels=len(r_m5.channels))
h5 = model.fit(net5, Xa[i], oa[i], pa[i], epochs=80, batch_size=512, lr=3e-3, seed=0)
op, pp = model.predict_rays(net5, Xa[i])
m5_dice = float(2*((op>0.5)&oa[i].astype(bool)).sum()/((op>0.5).sum()+oa[i].sum()+1e-8))
m5_acc = float(((pp>0.5) == pa[i].astype(bool)).mean())
print(f"M5  loss {h5[0]['loss']:.4f} -> {h5[-1]['loss']:.4f}")
print(f"M5  occupancy Dice(train) {m5_dice:.3f} | presence acc {m5_acc:.3f}")
print("CONG: Dice > 0.90 va acc > 0.90.")

## 5. Ma trận P0–P3 (plan §7 chuẩn hóa)

`P3` bỏ qua nếu chưa có OOF softmax. Mỗi run tự sinh `experiment_id` và bản ghi audit.

In [ ]:
RUNS = {}
for alias in ("P0", "P1", "P2", "P3"):
    kw = {"prob_source": X.DEFAULT_PROB_SOURCE} if alias == "P3" else {}
    run = X.from_plan(alias, CLS, seed=1, **kw)
    if "prob" in run.channels and not glob.glob(f"{PROB_DIR}/{CLS}/*.nii.gz"):
        print(f"{alias}: bo qua - chua co OOF softmax (muc 1b)"); continue
    res = train_eval(run)
    RUNS[alias] = res
    print(f"{alias} {run.experiment_id}")
    print(f"     kenh {run.channels} | mien {run.domain} | vai tro M8: {run.m8_role}")
    print(f"     val occ-Dice {res['val_occ_dice']:.3f} | presence F1 "
          f"{res['presence']['f1']:.3f} | absent recall {res['presence']['absent_recall']:.3f}")

## 6. M6 — negative control hướng tia (plan §3.5)

**Test phản bác chính.** Nếu hướng tùy ý cũng tốt ngang pháp tuyến thì lợi ích đến từ
"thêm một mạng nữa", KHÔNG phải từ hệ tọa độ.

In [ ]:
BEST = "P2" if "P2" in RUNS else sorted(RUNS)[-1]
base_run = RUNS[BEST]["run"]
m6 = {}
for mode in ("normal", "axial", "tangent", "random"):
    Xa, oa, pa = build_for(base_run, train_ids, seed=0, direction=mode)
    Xb, ob, pb = build_for(base_run, val_ids, seed=100, direction=mode)
    net = model.RayEncoder1D(in_channels=len(base_run.channels))
    model.fit(net, Xa, oa, pa, epochs=30, batch_size=4096, lr=3e-4, seed=1)
    op, pp = model.predict_rays(net, Xb)
    m6[mode] = {"occ_dice": float(2*((op>0.5)&ob.astype(bool)).sum()/((op>0.5).sum()+ob.sum()+1e-8)),
                "presence_f1": metrics.presence_f1(pb.astype(bool), pp>0.5)["f1"]}
    print(f"{mode:<9} occ-Dice {m6[mode]['occ_dice']:.3f}  presence F1 {m6[mode]['presence_f1']:.3f}")

ctrl = max(m6[k]["occ_dice"] for k in ("axial","tangent","random"))
print(f"\nM6: normal {m6['normal']['occ_dice']:.3f} vs doi chung tot nhat {ctrl:.3f}")
print("CONG: normal phai HON HAN. Neu khong => loi ich KHONG tu he toa do - phai bao cao that.")

## 7. M7 — surface jitter (plan §3.5) và P4/P5

M7 đo độ nhạy lúc *test*. P4 train **có** jitter. P5 dùng **xương dự đoán** (S2).
§4.9 đặt mốc: giữ **60–70%** lợi ích khi chuyển sang bề mặt dự đoán.

In [ ]:
# --- M7: jitter luc test ---
m7 = {}
for ds, dth in [(0.0, 0), (0.25, 5), (0.5, 10), (1.0, 15)]:
    Xb, ob, pb = build_for(base_run, val_ids, seed=100,
                           jitter_s_mm=ds, jitter_theta_deg=dth)
    op, pp = model.predict_rays(RUNS[BEST]["net"], Xb)
    m7[f"{ds}mm/{dth}deg"] = {
        "occ_dice": float(2*((op>0.5)&ob.astype(bool)).sum()/((op>0.5).sum()+ob.sum()+1e-8)),
        "presence_f1": metrics.presence_f1(pb.astype(bool), pp>0.5)["f1"]}
    print(f"jitter {ds:.2f}mm/{dth:>2}deg  occ-Dice {m7[f'{ds}mm/{dth}deg']['occ_dice']:.3f}")
print("Suy giam PHAI tu tu. Sup dot ngot => he toa do gion, xem lai truoc P5.\n")

# --- P4 (S1: train co jitter) va P5 (S2: xuong du doan) ---
BEST_I = base_run.inputs
for alias in ("P4", "P5"):
    kw = {"prob_source": X.DEFAULT_PROB_SOURCE} if BEST_I == "I3" else {}
    run = X.from_plan(alias, CLS, inputs=BEST_I, seed=1, **kw)
    if alias == "P5":
        miss = [c for c in train_ids + val_ids if baseline_path(c) is None]
        if miss:
            print(f"P5: bo qua - {len(miss)} ca thieu prediction xuong"); continue
    res = train_eval(run)
    RUNS[alias] = res
    print(f"{alias} {run.experiment_id}  val occ-Dice {res['val_occ_dice']:.3f} "
          f"| presence F1 {res['presence']['f1']:.3f}")

if "P5" in RUNS:
    keep = RUNS["P5"]["val_occ_dice"] / RUNS[BEST]["val_occ_dice"]
    print(f"\nP5 giu duoc {keep:.0%} hieu nang so voi xuong GT  (§4.9 moc 60-70%)")

## 8. Bảng ablation M8 (§8 — nhóm phân tích, không phải run mới)

M8 = input ablation dưới scaffold cố định **S0-D1-H1**. Vai trò được **suy ra** từ các run
đã train — không train lại. `assert_single_factor` chặn so sánh đổi >1 trục.

In [ ]:
scaffold = [r for r in RUNS.values() if r["run"].m8_role]
print(f"{'M8':<6}{'run':<8}{'kenh':<22}{'occ-Dice':>10}{'presence F1':>13}")
for r in sorted(scaffold, key=lambda r: r["run"].m8_role):
    run = r["run"]
    print(f"{run.m8_role:<6}{run.plan_alias:<8}{str(run.channels):<22}"
          f"{r['val_occ_dice']:>10.3f}{r['presence']['f1']:>13.3f}")

# Dong gop cua coarse prior: chi hop le neu CHI kenh thay doi
if "P3" in RUNS and "P2" in RUNS:
    try:
        X.assert_single_factor(RUNS["P2"]["run"], RUNS["P3"]["run"], "inputs")
        d = RUNS["P3"]["val_occ_dice"] - RUNS["P2"]["val_occ_dice"]
        print(f"\nDong gop coarse ResEnc prior (I2 -> I3): {d:+.3f} occ-Dice")
        print("Neu dong gop LON => mo hinh chu yeu HIEU CHINH ResEnc, khong tu doc MRI.")
        print("Do la rui ro that cho luan diem - phai bao cao (§3.5 M8).")
    except ValueError as e:
        print("\nKhong so sanh duoc:", e)

## 9. Đánh giá §3.7 — mẫu số VÙNG MỎNG (không phải ASSD tổng)

In [ ]:
EVAL_CKPT = f"{BSC_ROOT}/runs/mvp_eval_{CLS}_{BEST}.jsonl"
net = RUNS[BEST]["net"]
done = {json.loads(l)["case"] for l in open(EVAL_CKPT)} if os.path.exists(EVAL_CKPT) else set()

with open(EVAL_CKPT, "a") as fh:
    for cid in tqdm(val_ids, desc="eval"):
        if cid in done or baseline_path(cid) is None: continue
        gt, sp = io_utils.load_nii(f"{RAW}/labelsTr/{cid}.nii.gz")
        mri, _ = io_utils.load_nii(f"{RAW}/imagesTr/{cid}_0000.nii.gz")
        b0, _ = io_utils.load_nii(baseline_path(cid))
        bone, cart = (gt == BONE[CLS]), (gt == CART[CLS])
        if not bone.any() or not cart.any(): continue

        # FULL marching-cubes density khi inference (KHONG subsample) - xem core.py
        _, verts, dirs0 = core.bone_geometry(bone, sp, cfg)
        occ0 = core.occupancy_target(cart, verts, dirs0, cfg, sp)
        dom = domain_for(base_run, verts, occ0)
        Xc, occ, pres, v2, d2 = model.build_case_rays(
            mri, bone, cart, sp, cfg, domain=dom, channels=base_run.channels)
        op, pp = model.predict_rays(net, Xc)
        vol = model.reconstruct_volume(op, v2, d2, cart.shape, cfg, sp, presence_prob=pp)

        tf = headroom.gt_thickness_per_node(bone, cart, sp, cfg)
        fh.write(json.dumps({
            "case": cid,
            "thin_b0":  headroom.thin_region_boundary_error(cart, (b0 == CART[CLS]), bone, sp, cfg, tf),
            "thin_ray": headroom.thin_region_boundary_error(cart, vol > 0.5, bone, sp, cfg, tf),
            "presence": metrics.presence_f1(pres.astype(bool), pp > 0.5)}) + "\n")
        fh.flush()

rows = [json.loads(l) for l in open(EVAL_CKPT)]
b = np.array([r["thin_b0"]["thin_mean_err_mm"] for r in rows], float)
m = np.array([r["thin_ray"]["thin_mean_err_mm"] for r in rows], float)
ok_ = np.isfinite(b) & np.isfinite(m); b, m = b[ok_], m[ok_]
rel = (b - m) / b
boot = metrics.paired_bootstrap(m, b)

print(f"\nn = {len(b)} ca val (out-of-fold)")
print(f"loi bien vung mong   B0 {b.mean():.4f}mm  ->  ray {m.mean():.4f}mm")
print(f"cai thien tuong doi  {rel.mean():+.1%}   (muc tieu §3.7: >= +10%)")
print(f"cai thien tuyet doi  {boot['delta_mean']:+.4f}mm  CI [{boot['ci_low']:+.4f}, {boot['ci_high']:+.4f}]")
print(f"so ca tot hon        {boot['n_better']}/{boot['n']}")
print(f"presence F1 {np.mean([r['presence']['f1'] for r in rows]):.3f} | "
      f"absent recall {np.mean([r['presence']['absent_recall'] for r in rows]):.3f}")

## 10. Cổng §3.7 + ghi registry (§11–§12)

In [ ]:
gate = {
    "1_thin_boundary_rel_improve": float(rel.mean()),
    "1_pass_10pct": bool(rel.mean() >= 0.10),
    "1_ci_low_positive": bool(boot["ci_low"] > 0),
    "5_m6_normal_beats_controls": bool(m6["normal"]["occ_dice"] > ctrl),
    "m5_passed": bool(m5_dice > 0.90 and m5_acc > 0.90),
    "6_p5_gain_retained": (float(RUNS["P5"]["val_occ_dice"] / RUNS[BEST]["val_occ_dice"])
                           if "P5" in RUNS else None),
}
print(json.dumps(gate, indent=2, ensure_ascii=False))

registry = []
for alias, r in RUNS.items():
    rec = r["run"].to_registry(dataset_revision="Dataset001_KneeOA")
    rec.update({"val_occ_dice": r["val_occ_dice"], "presence": r["presence"],
                "n_train_rays": r["n_train_rays"], "n_train_cases": len(train_ids),
                "n_val_cases": len(val_ids)})
    registry.append(rec)

out = {"class": CLS, "registry": registry, "m6": m6, "m7": m7, "gate37": gate,
       "atlas": {"n_cases": len(ATLAS.case_ids), "fold": 0},
       "m5": {"occ_dice": m5_dice, "presence_acc": m5_acc}}
path = f"{BSC_ROOT}/runs/MVP_stage1_{CLS}.json"
json.dump(out, open(path, "w"), indent=2, ensure_ascii=False)
print("\nDa ghi", path)

print("""
NHAC LAI (M0 §6): KHONG bao cao ASSD tong - muc tieu 0.006mm nho ngang sai khac giua hai
implementation metric. Bao cao: loi bien VUNG MONG, presence F1, absent recall, thickness MAE.
M6 la test phan bac: khong co no thi ket qua duong tinh KHONG quy duoc cho he toa do.""")